Loading the parquet file + schema health check

In [1]:
import os
import shutil
import pandas as pd
import matplotlib as plt
import seaborn as sns
import pyarrow.parquet as pq
import fsspec
import time
import duckdb
from pathlib import Path
import polars as pl

In [4]:
urls = {
    "jan": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet",
    "feb": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet",
    "mar": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet",
    "apr": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-04.parquet",
    "may": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-05.parquet",
    "jun": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet",
    "jul": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet",
    "aug": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-08.parquet",
    "sep": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-09.parquet",
    "oct": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-10.parquet",
    "nov": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet",
    "dec": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-12.parquet",
}

schemas = {}

# schema health check
for name, url in urls.items():
    with fsspec.open(url, "rb") as f:
        pf = pq.ParquetFile(f)
        schemas[name] = pf.schema_arrow
    print(f"\n=== {name} ====")
    print(pf.schema)

ref_month, ref_schema = next(iter(schemas.items()))
bad = [m for m, s in schemas.items() if not s.equals(ref_schema)]
print("All match" if not bad else f" mismatch: {bad}")


=== jan ====
required group field_id=-1 schema {
  optional int32 field_id=-1 VendorID;
  optional int64 field_id=-1 tpep_pickup_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 tpep_dropoff_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 passenger_count;
  optional double field_id=-1 trip_distance;
  optional int64 field_id=-1 RatecodeID;
  optional binary field_id=-1 store_and_fwd_flag (String);
  optional int32 field_id=-1 PULocationID;
  optional int32 field_id=-1 DOLocationID;
  optional int64 field_id=-1 payment_type;
  optional double field_id=-1 fare_amount;
  optional double field_id=-1 extra;
  optional double field_id=-1 mta_tax;
  optional double field_id=-1 tip_amount;
  optional double field_id=-1 tolls_amount;
  optional double field_id=-1 improvement_s

One-time local combined parquet build

In [ ]:
local_parquet = Path("data/raw/yellow_2025_combined.parquet")
local_monthly_dir = Path("data/raw/monthly_2025")
DOWNLOAD_BUFFER_SIZE = 8 * 1024 * 1024
ROW_GROUP_SIZE = 500000

def materialize_monthly_parquets(urls_dict, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)

    local_paths = []
    t0 = time.perf_counter()

    for url in urls_dict.values():
        filename = url.rsplit("/", 1)[-1]
        target = local_dir / filename

        if not target.exists():
            tmp_target = target.with_suffix(target.suffix + ".tmp")

            try:
                with fsspec.open(url, "rb") as src, open(tmp_target, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=DOWNLOAD_BUFFER_SIZE)
                os.replace(tmp_target, target)
            except Exception:
                tmp_target.unlink(missing_ok=True)
                raise

        local_paths.append(target.as_posix())

    t1 = time.perf_counter()
    print(f"Monthly cache ready in {t1 - t0:.2f}s -> {local_dir}")
    return local_paths

def build_local_combined_parquet(urls_dict, output_path, monthly_dir):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        print(f"Already exists: {output_path}")
        return

    monthly_paths = materialize_monthly_parquets(urls_dict, monthly_dir)

    quoted_paths = ", ".join([f"'{p}'" for p in monthly_paths])

    t0 = time.perf_counter()
    # 500k balances build throughput and downstream scan granularity without large memory spikes during write.
    sql = f"""
    COPY (
        SELECT *
        FROM read_parquet([{quoted_paths}], union_by_name=false)
    )
    TO '{output_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE {ROW_GROUP_SIZE});
    """

    with duckdb.connect() as con:
        con.execute(f"PRAGMA threads={max(1, os.cpu_count() or 1)}")
        con.execute("SET preserve_insertion_order = false")
        con.execute(sql)

    t1 = time.perf_counter()
    print(f"Built once in {t1 - t0:.2f}s -> {output_path}")

build_local_combined_parquet(urls, local_parquet, local_monthly_dir)


Monthly cache ready in 1352.89s -> data\raw\monthly_2025
Built once in 18.92s -> data\raw\yellow_2025_combined.parquet


In [2]:
yellow_lf = pl.scan_parquet("data/raw/yellow_2025_combined.parquet")
yellow_lf.head(5).collect()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,1,"""N""",164,233,1,7.9,0.0,0.5,1.0,0.0,1.0,13.65,2.5,0.0,0.75
2,2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,1,"""N""",229,138,2,42.9,5.0,0.5,0.0,6.94,1.0,59.59,2.5,0.0,0.75
1,2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,1,"""N""",79,162,1,14.9,3.25,0.5,1.0,0.0,1.0,20.65,2.5,0.0,0.75
1,2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,1,"""N""",263,163,1,11.4,3.25,0.5,3.2,0.0,1.0,19.35,2.5,0.0,0.75
2,2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,1,"""N""",224,186,1,12.1,0.0,0.5,1.15,0.0,1.0,18.0,2.5,0.0,0.75


Lemme focus on some key columns...

In [3]:
yellow_key = yellow_lf.select(["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "PULocationID", "DOLocationID", "total_amount"])

In [4]:
print(yellow_key.describe())

shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ tpep_picku ┆ tpep_dropo ┆ passenger ┆ trip_dist ┆ PULocatio ┆ DOLocatio ┆ total_amo │
│ ---        ┆ p_datetime ┆ ff_datetim ┆ _count    ┆ ance      ┆ nID       ┆ nID       ┆ unt       │
│ str        ┆ ---        ┆ e          ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆ ---        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
│            ┆            ┆ str        ┆           ┆           ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 48722602   ┆ 48722602   ┆ 3.7110708 ┆ 4.8722602 ┆ 4.8722602 ┆ 4.8722602 ┆ 4.8722602 │
│            ┆            ┆            ┆ e7        ┆ e7        ┆ e7        ┆ e7        ┆ e7        │
│ null_count ┆ 0          ┆ 0          ┆ 1.1611894 ┆ 0.0       ┆ 0.0       ┆ 

Interesting... what has caught my eye is that I am seeing 2007 somewhere and now I am worried I got some wrong data yet I am certain that I got the right one.... even the links show the date... I am also seeing negatives... I need to find out how to work with these values!

In [ ]:
exists_2007 = (
    yellow_lf.select(
        (pl.col("tpep_pickup_datetime").dt.year() == 2007).any().alias("exists_2007")
    )
    .collect()
    .item()
)
exists_2007

True

Oh dear... now this needs me to pause and figure out where this 2007 is coming from

In [ ]:
filter_2007 = (
    yellow_lf.filter(pl.col("tpep_pickup_datetime").dt.year() == 2007)
    .limit(20)
    .collect()
)
filter_2007

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2007-12-05 18:45:00,2007-12-05 19:02:00,1,3.0,1,"""N""",142,234,2,17.0,1.0,0.5,0.0,0.0,1.0,22.75,2.5,0.0,0.75


God knows how that row snuck in there! Super random! There may be more though... lemme figure out which years are used in the whole column

In [ ]:
unique_years = (
    yellow_lf.select(pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"))
    .unique()
    .sort("pickup_year")
    .collect()
)
unique_years

pickup_year
i32
2007
2008
2009
2024
2025


Great... there's more of them! Meanwhile my conda kernel can't handle this workload... it keeps crashing so I may need to switch to something else soon! Or let's try polars

In [ ]:
filter_2008 = (
    yellow_lf.filter(pl.col("tpep_pickup_datetime").dt.year() == 2008)
    .limit(20)
    .collect()
)
filter_2009 = (
    yellow_lf.filter(pl.col("tpep_pickup_datetime").dt.year() == 2009)
    .limit(20)
    .collect()
)
filter_2024 = (
    yellow_lf.filter(pl.col("tpep_pickup_datetime").dt.year() == 2024)
    .limit(20)
    .collect()
)

In [13]:
filter_2008

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2008-12-31 23:04:21,2008-12-31 23:32:25,1,18.12,2,"""N""",132,238,2,70.0,0.0,0.5,0.0,6.94,1.0,80.94,2.5,0.0,0.0


In [12]:
filter_2009

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2009-01-01 00:19:34,2009-01-01 01:10:21,6,10.77,1,"""N""",138,239,2,52.7,5.0,0.5,0.0,6.94,1.0,70.39,2.5,1.75,0.0
2,2009-01-01 00:20:39,2009-01-01 00:20:49,5,4.47,1,"""N""",230,261,1,24.0,1.0,0.5,2.98,0.0,1.0,32.73,2.5,0.0,0.75
2,2009-01-01 08:52:26,2009-01-01 10:00:26,1,11.95,1,"""N""",138,163,1,64.6,0.0,0.5,16.65,13.88,1.0,99.88,2.5,0.0,0.75
2,2009-01-01 12:52:15,2009-01-01 13:12:15,1,2.13,1,"""N""",43,230,1,18.4,0.0,0.5,4.63,0.0,1.0,27.78,2.5,0.0,0.75
2,2009-01-01 14:08:04,2009-01-01 14:45:04,1,11.56,1,"""N""",138,142,2,50.6,0.0,0.5,0.0,13.88,1.0,68.48,2.5,0.0,0.0
2,2009-01-01 00:05:50,2009-01-01 00:10:56,1,1.26,1,"""N""",239,50,2,8.6,1.0,0.5,0.0,0.0,1.0,14.35,2.5,0.0,0.75


In [11]:
filter_2024

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-12-31 23:30:03,2024-12-31 23:43:02,1,3.0,1,"""N""",246,13,1,16.3,1.0,0.5,5.32,0.0,1.0,26.62,2.5,0.0,0.0
2,2024-12-31 23:31:38,2024-12-31 23:41:48,1,1.03,1,"""N""",43,140,2,10.7,1.0,0.5,0.0,0.0,1.0,15.7,2.5,0.0,0.0
2,2024-12-31 23:46:38,2025-01-01 00:03:03,1,3.95,1,"""N""",229,24,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0
2,2024-12-31 23:56:19,2025-01-01 00:11:19,6,2.28,1,"""N""",68,107,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.0,0.0
2,2024-12-31 23:55:37,2025-01-01 00:01:26,1,1.12,1,"""N""",56,56,2,7.9,1.0,0.5,0.0,0.0,1.0,10.4,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2024-12-31 20:47:55,2024-12-31 20:54:00,2,1.72,1,"""N""",48,246,1,9.3,1.0,0.5,2.86,0.0,1.0,17.16,2.5,0.0,0.0
2,2024-12-31 20:54:50,2024-12-31 21:30:18,2,1.39,1,"""N""",246,48,1,28.2,1.0,0.5,6.64,0.0,1.0,39.84,2.5,0.0,0.0
2,2024-12-31 21:20:05,2024-12-31 21:35:13,2,2.64,1,"""N""",42,238,1,16.3,1.0,0.5,2.0,0.0,1.0,23.3,2.5,0.0,0.0


Well this is manageable... I'll just need to remove the rows based on the filters I have made... However interestingly, I hadn't considered/thought about the edge case of someone travelling when it's just before the new year and hence they end up elsewhere in another year... those ones I guess I may have to think it through... Alright I contacted DK and so I'll just delete the overlaps, and ignore the fare column becuase after all... it may not be a predictor

In [5]:
invalid_years = [2007, 2008, 2009, 2024]
yellow_df_cleaned = yellow_key.filter(
    ~pl.col("tpep_pickup_datetime").dt.year().is_in(invalid_years)
)

In [6]:
yellow_df_cleaned.head(5).collect()

tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,total_amount
datetime[μs],datetime[μs],i64,f64,i32,i32,f64
2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,164,233,13.65
2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,229,138,59.59
2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,79,162,20.65
2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,263,163,19.35
2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,224,186,18.0


Let's now check the stats of our cleaned df...

In [7]:
print(yellow_df_cleaned.describe())

shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ tpep_picku ┆ tpep_dropo ┆ passenger ┆ trip_dist ┆ PULocatio ┆ DOLocatio ┆ total_amo │
│ ---        ┆ p_datetime ┆ ff_datetim ┆ _count    ┆ ance      ┆ nID       ┆ nID       ┆ unt       │
│ str        ┆ ---        ┆ e          ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆ ---        ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
│            ┆            ┆ str        ┆           ┆           ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 48722573   ┆ 48722573   ┆ 3.7110679 ┆ 4.8722573 ┆ 4.8722573 ┆ 4.8722573 ┆ 4.8722573 │
│            ┆            ┆            ┆ e7        ┆ e7        ┆ e7        ┆ e7        ┆ e7        │
│ null_count ┆ 0          ┆ 0          ┆ 1.1611894 ┆ 0.0       ┆ 0.0       ┆ 

This data is quite funny... How does someone begin a trip in 2025 and is dropped off in 2024?! Guess I'll have to look into that as well...

In [9]:
filter_2024_dropoff = (
    yellow_df_cleaned.filter(pl.col("tpep_dropoff_datetime").dt.year() == 2024)
    .limit(20)
    .collect()
)
filter_2024_dropoff

tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,total_amount
datetime[μs],datetime[μs],i64,f64,i32,i32,f64
2025-01-23 01:44:59,2024-12-18 07:52:40,0,3.1,162,238,30.4


So it is indeed!... I'll have to remove it but now I'm thinking... What other dropoff times are "behind" the pickup time? Lemme see how I could do it... but first... let's remove that row... actually let's see all of the others if any

In [15]:
yellow_df_behind = (yellow_key.filter(pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime")).collect())
print(len(yellow_df_behind))

2235


Wow! so many wrong entries for time... could they be switched though?

In [18]:
print(yellow_df_behind.select(["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance"]).limit(20))

shape: (20, 3)
┌──────────────────────┬───────────────────────┬───────────────┐
│ tpep_pickup_datetime ┆ tpep_dropoff_datetime ┆ trip_distance │
│ ---                  ┆ ---                   ┆ ---           │
│ datetime[μs]         ┆ datetime[μs]          ┆ f64           │
╞══════════════════════╪═══════════════════════╪═══════════════╡
│ 2025-03-31 14:30:00  ┆ 2025-03-31 14:21:08   ┆ 4.4           │
│ 2025-03-02 09:03:39  ┆ 2025-03-02 09:03:24   ┆ 5.19          │
│ 2025-03-02 09:03:29  ┆ 2025-03-02 09:03:11   ┆ 6.61          │
│ 2025-03-03 06:03:39  ┆ 2025-03-03 06:03:25   ┆ 19.29         │
│ 2025-03-03 10:03:46  ┆ 2025-03-03 10:03:17   ┆ 0.76          │
│ …                    ┆ …                     ┆ …             │
│ 2025-03-06 16:03:44  ┆ 2025-03-06 16:03:06   ┆ 3.66          │
│ 2025-03-07 10:03:48  ┆ 2025-03-07 10:03:20   ┆ 5.22          │
│ 2025-03-08 00:03:37  ┆ 2025-03-08 00:03:21   ┆ 11.47         │
│ 2025-03-08 22:03:53  ┆ 2025-03-08 22:03:13   ┆ 10.21         │
│ 2025-03-

From the small sample, Some seem like a reasonable switchup... but others don't make sense realistically... I guess what I'll do is switch up some and delete the others... The ones that don't make sense seem to have a pattern... the "micro-seconds" part seems to be the only one changing... sure you could make the argument that the "micro-seconds" part and "minute" part got switched up for the dropoff time but that doesn't seem to be a reliable argument from the sample i have taken a look at.

In [19]:
# Swap pickup/dropoff only when times are reversed AND hour/minute changed.
is_behind = pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime")
hour_changed = pl.col("tpep_pickup_datetime").dt.hour() != pl.col("tpep_dropoff_datetime").dt.hour()
minute_changed = pl.col("tpep_pickup_datetime").dt.minute() != pl.col("tpep_dropoff_datetime").dt.minute()
swap_condition = is_behind & (hour_changed | minute_changed)

yellow_df_swapped = yellow_df_cleaned.with_columns(
    [
        pl.when(swap_condition)
        .then(pl.col("tpep_dropoff_datetime"))
        .otherwise(pl.col("tpep_pickup_datetime"))
        .alias("tpep_pickup_datetime"),
        pl.when(swap_condition)
        .then(pl.col("tpep_pickup_datetime"))
        .otherwise(pl.col("tpep_dropoff_datetime"))
        .alias("tpep_dropoff_datetime"),
    ]
)

rows_swapped = yellow_df_cleaned.filter(swap_condition).select(pl.len()).collect().item()
rows_still_behind = (
    yellow_df_swapped
    .filter(pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime"))
    .select(pl.len())
    .collect()
    .item()
)

print(f"Rows swapped: {rows_swapped}")
print(f"Rows still behind after swap: {rows_still_behind}")
yellow_df_swapped.head(5).collect()

Rows swapped: 1484
Rows still behind after swap: 751


tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,total_amount
datetime[μs],datetime[μs],i64,f64,i32,i32,f64
2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,164,233,13.65
2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,229,138,59.59
2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,79,162,20.65
2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,263,163,19.35
2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,224,186,18.0


In [21]:
# Drop any rows that are still temporally invalid after the swap step.
still_behind_condition = pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime")

rows_before = yellow_df_swapped.select(pl.len()).collect().item()
rows_to_drop = yellow_df_swapped.filter(still_behind_condition).select(pl.len()).collect().item()

yellow_df_final = yellow_df_swapped.filter(~still_behind_condition)

rows_after = yellow_df_final.select(pl.len()).collect().item()
rows_still_behind_final = yellow_df_final.filter(still_behind_condition).select(pl.len()).collect().item()

print(f"Rows before drop/filter: {rows_before}")
print(f"Rows dropped: {rows_to_drop}")
print(f"Rows after drop/filter: {rows_after}")
print(f"Rows still behind: {rows_still_behind_final}")

yellow_df_final.head(5).collect()

Rows before drop/filter: 48722573
Rows dropped: 751
Rows after drop/filter: 48721822
Rows still behind: 0


tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,total_amount
datetime[μs],datetime[μs],i64,f64,i32,i32,f64
2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,164,233,13.65
2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,229,138,59.59
2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,79,162,20.65
2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,263,163,19.35
2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,224,186,18.0


Now for those negative fares...

In [10]:
dropoff_time_behind = (
    yellow_lf.select(
        (pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime")).any().alias("dropoff_time_behind")
    )
    .collect()
    .item()
)
dropoff_time_behind

True

In [ ]:

schema = yellow_df_cleaned.collect_schema()
null_counts = yellow_df_cleaned.select(pl.all().null_count()).collect()
schema, null_counts

(Schema([('VendorID', Int32),
         ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
         ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
         ('passenger_count', Int64),
         ('trip_distance', Float64),
         ('RatecodeID', Int64),
         ('store_and_fwd_flag', String),
         ('PULocationID', Int32),
         ('DOLocationID', Int32),
         ('payment_type', Int64),
         ('fare_amount', Float64),
         ('extra', Float64),
         ('mta_tax', Float64),
         ('tip_amount', Float64),
         ('tolls_amount', Float64),
         ('improvement_surcharge', Float64),
         ('total_amount', Float64),
         ('congestion_surcharge', Float64),
         ('Airport_fee', Float64),
         ('cbd_congestion_fee', Float64)]),
 shape: (1, 20)
 ┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ VendorID ┆ tpep_pick ┆ tpep_drop ┆ passenger ┆ … ┆ total_amo ┆ congestio ┆ A